# 18: Paralelismo com Send (Analyzer e Investigator juntos)

Quando uma anomalia nos números da empresa era detectada, a análise seguia um caminho estritamente sequencial:

1. Supervisor -> YFinance
2. Supervisor -> Analyzer (notícias da empresa)
3. Supervisor -> Analyzer (contexto do setor)
4. Supervisor -> Investigator (diagnóstico causal)
5. Supervisor -> Consolidador

Cada etapa aguardava a anterior terminar para só então iniciar a seguinte. O tempo total era a soma direta
de todas as chamadas: a coleta de mercado (~3 s), as notícias (~10 s), o setor (~10 s), a investigação (~15 s)
e a consolidação (~5 s), somando mais de 40 segundos e 5 passagens pelo Supervisor.

No entanto, depois que o YFinance entrega os dados de mercado e os sinais de anomalia, o trabalho do Analyzer
(notícias e setor) e o do Investigator (diagnóstico causal da anomalia) não dependem um do outro. Esperar o
Analyzer concluir para só depois iniciar o Investigator acumulava tempo de espera desnecessário.

A solução é disparar os dois agentes ao mesmo tempo usando a primitiva `Send` do LangGraph. O tempo dessa etapa
passa a ser o tempo do agente mais lento, e o ciclo do coordenador cai de 5 para 3 voltas.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Conceitos

**O que é o `Send`.** Em grafos comuns, a função de roteamento devolve apenas o nome do próximo nó como string.
Com a classe `Send("nome_do_no", estado_customizado)`, o roteador pode devolver uma lista de destinos. Cada nó
indicado é agendado de forma concorrente, recebendo o estado montado especificamente para ele.

**Superstep no LangGraph.** A execução do grafo ocorre em supersteps. Todos os nós agendados para a mesma rodada
executam concorrentemente. O superstep só termina quando todos os nós dessa rodada concluem. No final do passo,
as atualizações de todos eles são aplicadas juntas ao estado.

**Por que os dois ramos voltam ao Supervisor e ele roda uma vez só.** Tanto a aresta do Analyzer quanto a do
Investigator apontam de volta para o nó do Supervisor:

```python
workflow.add_edge("Analyzer", "Supervisor")
workflow.add_edge("Investigator", "Supervisor")
```

Como os dois agentes terminam no mesmo superstep e ambos têm como próximo destino o Supervisor, o LangGraph
detecta a convergência e agenda o Supervisor uma única vez no superstep seguinte. Não é necessário usar `defer`
nesta fase.

**Reducers que aceitam escrita concorrente (sem colisão):**

| Campo no state | Reducer | Comportamento concorrente |
|---|---|---|
| `messages` | `add_messages` | Concatena: as mensagens de ambos entram no histórico sem perdas |
| `erros` | `operator.or_` | Une dicionários: Analyzer atualiza sua chave e Investigator a dele |
| `tentativas` | `operator.or_` | Une dicionários: chaves disjuntas combinam perfeitamente |
| `noticias`, `contexto_setor` | padrão (sobrescreve) | Escrito apenas pelo Analyzer; Investigator não toca |
| `diagnostico` | padrão (sobrescreve) | Escrito apenas pelo Investigator; Analyzer não toca |
| `loop_count` | `operator.add` | Nenhum dos workers incrementa voltas; apenas o Supervisor soma 1 |

**A estratégia híbrida de roteamento.** Decidir se investiga não precisa ser uma escolha exclusiva entre regras
rígidas ou liberdade total do modelo:

1. **Com sinais de anomalia:** Se o detector numérico encontra queda de lucro acima de 30% contra o mesmo período
   do ano anterior, prejuízo ou Piotroski fraco, o guard da Etapa 2 escolhe `next_agent = PARALELO`. O roteador
   despacha os dois `Send` imediatamente, sem chamada ao LLM.
2. **Sem sinais de anomalia:** O Analyzer executa sozinho. Na Etapa 3, o LLM do Supervisor examina todo o contexto
   e decide se há alguma particularidade que justifique acionar o Investigator ou se segue para `FINISH`.

## 2. O `router` com estado montado à mão

Testamos a função `router` com um estado preparado em memória. Nenhuma chamada de rede acontece nesta célula.
O foco é conferir se a decisão `PARALELO` produz dois objetos `Send` e se a função `historico_do_ramo` isola a
tarefa de cada agente.

In [2]:
from langchain_core.messages import HumanMessage
from langgraph.types import Send

from fluxo_financeiro.graph.supervisor import (
    PARALELO,
    TAREFA_ANALISTA,
    _tarefa_investigador,
    router,
)
from fluxo_financeiro.shared.types import SinalAnomalia

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage("Dados para VALE3.SA: P/L 5.2x, lucro caiu 35%...", name="YFinance")
sinal = SinalAnomalia(tipo="queda_lucro", descricao="Lucro líquido caiu 35% contra o mesmo trimestre do ano anterior.")

tarefa_ana = HumanMessage(content=TAREFA_ANALISTA([pedido, dados]), name="Supervisor")
tarefa_inv = HumanMessage(content=_tarefa_investigador([sinal]), name="Supervisor")

estado = {
    "messages": [pedido, dados, tarefa_ana, tarefa_inv],
    "next_agent": PARALELO,
    "sinais_anomalia": [sinal],
    "dados_mercado": dados.content,
    "loop_count": 1,
}

sends = router(estado)

print(f"Tipo retornado: {type(sends).__name__}")
print(f"Quantidade de destinos: {len(sends)}")
for s in sends:
    print(f"   -> Nó: {s.node} (tipo {type(s).__name__})")

2026-11-09 13:49:16,896 - fluxo_financeiro - DEBUG - [router] paralelo → Analyzer + Investigator


Tipo retornado: list
Quantidade de destinos: 2
   -> Nó: Analyzer (tipo Send)
   -> Nó: Investigator (tipo Send)


### Isolamento de histórico em cada ramo

Se o histórico fosse passado por inteiro, o Analyzer leria a tarefa do Investigator (a última mensagem do Supervisor)
e acharia que sua função é investigar a anomalia. `historico_do_ramo` filtra a instrução do outro agente.

In [3]:
msgs_ana = sends[0].arg["messages"]
msgs_inv = sends[1].arg["messages"]

print("--- Ramo Analyzer ---")
print("Total de mensagens:", len(msgs_ana))
print("Última mensagem:", msgs_ana[-1].name, "->", msgs_ana[-1].content.splitlines()[0])
tem_tarefa_inv = any(sinal.descricao in m.content for m in msgs_ana)
print("Contém a tarefa do Investigator?", tem_tarefa_inv)

print("\n--- Ramo Investigator ---")
print("Total de mensagens:", len(msgs_inv))
print("Última mensagem:", msgs_inv[-1].name, "->", msgs_inv[-1].content.splitlines()[0])
tem_tarefa_ana = any("notícias recentes" in m.content for m in msgs_inv)
print("Contém a tarefa do Analyzer?", tem_tarefa_ana)

--- Ramo Analyzer ---
Total de mensagens: 3
Última mensagem: Supervisor -> Busque notícias recentes sobre a empresa no histórico. Use os dados financeiros do YFinance como contexto para identificar o ticker e o nome da empresa.
Contém a tarefa do Investigator? False

--- Ramo Investigator ---
Total de mensagens: 3
Última mensagem: Supervisor -> Investigue as anomalias detectadas nos números da empresa:
Contém a tarefa do Analyzer? False


## 3. Uma análise real da VALE3 com `app.stream`

Executamos o pipeline completo para a ação VALE3. Como ela apresenta sinal de anomalia contábil, o guard
da Etapa 2 encaminha para o modo paralelo. Acompanhamos passo a passo a execução concorrente e medimos
o tempo total decorrido.

In [4]:
import time
from langchain_core.messages import HumanMessage
from fluxo_financeiro import app

print("Iniciando execução da VALE3...")
inicio = time.perf_counter()

passos = []
final = None

for modo, pedaco in app.stream({"messages": [HumanMessage("Analise a ação VALE3")]}, stream_mode=["updates", "values"]):
    if modo == "values":
        final = pedaco
        continue
    for no, update in pedaco.items():
        update = update or {}
        if no == "Supervisor":
            proximo = update.get("next_agent")
            decisao = update.get("decidido_por")
            print(f"[Supervisor] -> {proximo} (decidido por: {decisao})")
            if update.get("supervisor_reasoning"):
                print(f"   motivo: {update['supervisor_reasoning'][:140]}...")
            passos.append(f"Supervisor->{proximo}")
        else:
            tam = len(update.get("messages", [])[-1].text) if update.get("messages") else 0
            print(f"[{no}] resposta recebida ({tam} caracteres)")
            passos.append(no)

duracao = time.perf_counter() - inicio

print("\n" + "=" * 60)
print("Caminho percorrido:", " | ".join(passos))
print(f"Tempo total decorrido: {duracao:.2f} s")
print("=" * 60)

2026-11-09 13:49:16,947 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


Iniciando execução da VALE3...


2026-11-09 13:49:18,385 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços, balanço e indicadores fundamentalistas para a ação VALE3.'


2026-11-09 13:49:18,386 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-09 13:49:18,388 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


[Supervisor] -> YFinance (decidido por: llm)
   motivo: O usuário solicitou a análise da ação VALE3. Como ainda não há mensagens do YFinance no histórico, o próximo passo obrigatório é coletar os ...


... (36 outputs left out)

2026-11-09 13:49:34,711 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (7 msgs)


[Analyzer] resposta recebida (2724 caracteres)
[Supervisor] -> FINISH (decidido por: guard)


2026-11-09 13:49:37,232 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (3341 chars)


[Consolidador] resposta recebida (3341 caracteres)

Caminho percorrido: Supervisor->YFinance | YFinance | Supervisor->Analyzer+Investigator | Investigator | Analyzer | Supervisor->FINISH | Consolidador
Tempo total decorrido: 20.32 s


## 4. O state final e os campos preenchidos

Conferimos como o estado final acomodou as respostas produzidas em paralelo. Cada agente preencheu seus campos
próprios sem interferir nas demais informações.

In [5]:
for campo in ("dados_mercado", "noticias", "contexto_setor", "diagnostico"):
    valor = final.get(campo)
    tamanho = f"{len(valor)} caracteres" if valor else "None"
    print(f"{campo:16}: {tamanho}")

print()
print("loop_count final:", final.get("loop_count"))
print("erros:           ", final.get("erros"))
print("tentativas:      ", final.get("tentativas"))
relatorio_ok = final.get("relatorio") is not None
print("relatório estruturado pronto:", relatorio_ok)
if relatorio_ok and final["relatorio"].diagnostico:
    print("diagnóstico copiado no relatório final: sim")

dados_mercado   : 3588 caracteres
noticias        : 1369 caracteres
contexto_setor  : 1309 caracteres
diagnostico     : 1248 caracteres

loop_count final: 3
erros:            {'YFinance': None, 'Analyzer': None, 'Investigator': None}
tentativas:       {'YFinance': 0, 'Analyzer': 0, 'Investigator': 0}
relatório estruturado pronto: True
diagnóstico copiado no relatório final: sim


## O que observar

- Na chamada com estado montado, o router retornou uma lista de objetos Send com nós destinos Analyzer e Investigator.
- A função historico_do_ramo manteve cada tarefa separada: o Analyzer não recebeu as orientações de investigação de anomalia, e o Investigator não recebeu o pedido de notícias e setor.
- Na execução real da VALE3, o guard da Etapa 2 identificou os sinais de anomalia e despachou Analyzer e Investigator no mesmo superstep.
- O tempo total da etapa conjunta reflete a duração do agente mais demorado, em vez de somar sequencialmente o tempo de ambos.
- Ambos os ramos retornaram ao Supervisor pela aresta padrão do grafo. O LangGraph sincronizou a conclusão e acionou o Supervisor uma única vez.
- O ciclo completo da análise com anomalia encerrou com loop_count em 3, contra as 5 voltas da arquitetura sequencial anterior.
- A concorrência não gerou conflitos no estado: messages utilizou add_messages, erros e tentativas operaram com operator.or_, e noticias, contexto_setor e diagnostico ocuparam campos isolados.
- A decisão híbrida atuou sem intervenção manual: a presença de sinais disparou o paralelo de forma determinística por guard, poupando chamadas de LLM.